In [96]:
import numpy as np
import pandas as pd
import tkinter as tk
import tkinter.font as tkFont
import joblib

In [97]:
model = joblib.load('model.joblib')
genomic_pp = joblib.load('genomic_pp.joblib')
clinical_pp = joblib.load('clinical_pp.joblib')
laboratory_pp = joblib.load('laboratory_pp.joblib')
treatment_pp = joblib.load('treatment_pp.joblib')

In [98]:
root = tk.Tk()
root.title('Oncology Patient Monitor')
root.configure(bg = '#0d1117')

In [99]:
default_font = tkFont.nametofont("TkDefaultFont")
default_font.configure(family="Segoe UI", size=10)
root.option_add("*Font", default_font)

In [100]:
default = 'Choose'

In [101]:
def predict():
    try:
        genomic = pd.DataFrame({
            'Mutation_TP53': [int(a1.get())],
            'Mutation_IDH1': [int(a2.get())],
            'Mutation_EGFR': [int(a3.get())],
            'Gene_1': [float(b1.get())],
            'Gene_2': [float(b2.get())],
            'Gene_3': [float(b3.get())],
            'Gene_4': [float(b4.get())],
            'Gene_5': [float(b5.get())]
        })
    
        for i in range(1, 6):
            genomic[f'Gene_{i}'] = np.log2(genomic[f'Gene_{i}'] + 1)
    
        clinical = pd.DataFrame({
            'Age': [int(c1.get())],
            'Gender': [c2.get()],
            'KPS': [int(c3.get())],
            'Comorbidities': [c4.get()]
        })
    
        laboratory = pd.DataFrame({
            'WBC': [int(d1.get())],
            'CRP': [float(d2.get())],
            'Albumin': [float(d3.get())]
        })
    
        treatment = pd.DataFrame({
            'Drug': [e1.get()],
            'Dose': [float(e2.get())],
            'Route': [e3.get()]
        })
    
        genomic = genomic_pp.transform(genomic)
        clinical = clinical_pp.transform(clinical)
        laboratory = laboratory_pp.transform(laboratory)
        treatment = treatment_pp.transform(treatment)
    
        pred = model.predict([genomic, clinical, laboratory, treatment])
    
        survival_prob = pred[0][0][0]
        response_probs = pred[1][0]
        toxicity_prob = pred[2][0][0]
        
        response_classes = ['CR', 'PD', 'PR', 'SD']
        response_class = response_classes[response_probs.argmax()]
        response_confidence = response_probs.max()
    
        if survival_prob >= 0.5:
            survival_status = 'high risk of an event within 24 months'
        else:
            survival_status = 'low risk of an event within 24 months'
    
        if toxicity_prob >= 0.5:
            toxicity_status = 'high risk of severe toxicity'
        else:
            toxicity_status = 'low risk of severe toxicity'
    
        result_label.config(text = f"Patient shows {survival_status},\npredicted treatment response of {response_class}\nand {toxicity_status}.", fg = '#e6edf3')

    except ValueError:
        result_label.config(text="Please enter right inputs.", fg = '#f85149')

In [102]:
title_frame = tk.Frame(root, bg = '#0a1628', height=70)
title_frame.pack(fill = 'x', pady = (0, 10))
title_frame.pack_propagate(False)

In [103]:
tk.Label(title_frame, text = 'ONCOLOGY PATIENT MONITOR',
         font = ('Segoe UI', 14, 'bold'),
         bg = '#0a1628', fg = '#e6edf3').pack(pady=18)

In [104]:
main_frame = tk.Frame(root, bg = '#161b22', relief = 'groove', bd = 1)
main_frame.pack(padx = 15, pady = 10, fill = 'both', expand = True)

In [105]:
frame_mutations = tk.Frame(main_frame, bg = '#161b22', width = 180)
frame_mutations.pack(side = 'left', padx = 12, pady = 12, fill = 'both', expand = True)

frame_genes = tk.Frame(main_frame, bg = '#161b22')
frame_genes.pack(side = 'left', padx = 12, pady = 12, fill = 'both', expand = True)

frame_clinical = tk.Frame(main_frame, bg = '#161b22')
frame_clinical.pack(side = 'left', padx = 12, pady = 12, fill = 'both', expand = True)

frame_laboratory = tk.Frame(main_frame, bg = '#161b22')
frame_laboratory.pack(side = 'left', padx = 12, pady = 12, fill = 'both', expand = True)

frame_treatment = tk.Frame(main_frame, bg = '#161b22')
frame_treatment.pack(side = 'left', padx = 12, pady = 12, fill = 'both', expand = True)

In [106]:
tk.Label(frame_mutations, text = 'MUTATIONS',
         font = ('Segoe UI', 10, 'bold'),
         bg = '#161b22', fg = '#58a6ff').pack(pady = (0, 8))

tk.Label(frame_genes, text = 'GENES',
         font = ('Segoe UI', 10, 'bold'),
         bg = '#161b22', fg = '#58a6ff').pack(pady = (0, 8))

tk.Label(frame_clinical, text = 'CLINICAL',
         font = ('Segoe UI', 10, 'bold'),
         bg = '#161b22', fg = '#58a6ff').pack(pady = (0, 8))

tk.Label(frame_laboratory, text="LABORATORY",
         font = ('Segoe UI', 10, 'bold'),
         bg = '#161b22', fg = '#58a6ff').pack(pady = (0, 8))

tk.Label(frame_treatment, text = 'TREATMENT',
         font = ('Segoe UI', 10, 'bold'),
         bg = '#161b22', fg = '#58a6ff').pack(pady = (0, 8))

In [107]:
tk.Label(frame_mutations, text = 'TP53 Mutation (0/1):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
a1 = tk.Entry(frame_mutations, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
a1.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_mutations, text = 'IDH1 Mutation (0/1):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
a2 = tk.Entry(frame_mutations, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
a2.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_mutations, text = 'EGFR Mutation (0/1):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
a3 = tk.Entry(frame_mutations, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
a3.pack(fill = 'x', pady = (0, 2))

In [108]:
tk.Label(frame_genes, text = 'Gene 1 (0-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
b1 = tk.Entry(frame_genes, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
b1.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_genes, text = 'Gene 2 (0-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
b2 = tk.Entry(frame_genes, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
b2.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_genes, text = 'Gene 3 (0-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
b3 = tk.Entry(frame_genes, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
b3.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_genes, text = 'Gene 4 (0-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
b4 = tk.Entry(frame_genes, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
b4.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_genes, text = 'Gene 5 (0-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
b5 = tk.Entry(frame_genes, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
b5.pack(fill = 'x', pady = (0, 2))

In [109]:
tk.Label(frame_clinical, text = 'Age (18-80):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
c1 = tk.Entry(frame_clinical, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
c1.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_clinical, text = 'Gender:', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
c2 = tk.StringVar(value = default)
gender_menu = tk.OptionMenu(frame_clinical, c2, 'Male', 'Female')
gender_menu.config(bg = '#0d1117', fg = '#e6edf3', relief = 'solid', bd = 1, highlightthickness = 0, padx = 4, pady = 1)
gender_menu.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_clinical, text = 'KPS (40-100):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
c3 = tk.Entry(frame_clinical, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
c3.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_clinical, text = 'Comorbidities:', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
c4 = tk.StringVar(value = default)
com_menu = tk.OptionMenu(frame_clinical, c4, 'None', 'Diabetes', 'Hypertension', 'Heart Disease', 'Multiple')
com_menu.config(bg = '#0d1117', fg = '#e6edf3', relief = 'solid', bd = 1, highlightthickness = 0, padx = 4, pady = 1)
com_menu.pack(fill = 'x', pady = (0, 2))

In [110]:
tk.Label(frame_laboratory, text = 'WBC (3500-15000):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
d1 = tk.Entry(frame_laboratory, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
d1.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_laboratory, text = 'CRP (0-30 mg/L):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
d2 = tk.Entry(frame_laboratory, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
d2.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_laboratory, text = 'Albumin (2.5-5.0):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
d3 = tk.Entry(frame_laboratory, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
d3.pack(fill = 'x', pady = (0, 2))

In [111]:
tk.Label(frame_treatment, text = 'Drug:', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
e1 = tk.StringVar(value = default)
drug_menu = tk.OptionMenu(frame_treatment, e1, 'Temozolomide', 'Lomustine', 'Bevacizumab', 'Carmustine', 'PCV')
drug_menu.config(bg = '#0d1117', fg = '#e6edf3', relief = 'solid', bd = 1, highlightthickness = 0, padx = 4, pady = 1)
drug_menu.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_treatment, text = 'Dose (50-300 mg):', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
e2 = tk.Entry(frame_treatment, relief = 'solid', bd = 1, bg = '#0d1117', fg = '#e6edf3', insertbackground = 'white')
e2.pack(fill = 'x', pady = (0, 2))

tk.Label(frame_treatment, text = 'Route:', bg = '#161b22', anchor = 'w', fg = '#c9d1d9').pack(fill = 'x', pady = (2, 0))
e3 = tk.StringVar(value = default)
route_menu = tk.OptionMenu(frame_treatment, e3, 'Oral', 'IV', 'Intrathecal')
route_menu.config(bg = '#0d1117', fg = '#e6edf3', relief = 'solid', bd = 1, highlightthickness = 0, padx = 4, pady = 1)
route_menu.pack(fill = 'x', pady = (0, 2))

In [112]:
button_frame = tk.Frame(root, bg = '#0d1117')
button_frame.pack(pady = 15)

tk.Button(button_frame, text = 'PREDICT',
          command = predict, bg = '#1f6feb', fg = '#ffffff', padx = 45, pady = 12,
          relief = 'raised', bd = 1, cursor = 'hand2',
          font = ('Segoe UI', 11, 'bold')).pack()

In [113]:
result_label = tk.Label(root, text = '', bg = '#0d1117',
                         font = ('Segoe UI', 12),
                         fg = '#e6edf3', justify = 'center')
result_label.pack(pady=10)

In [ ]:
root.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
